In [1]:
import torch
import torch.nn as nn
import torch.optim as optim

def load_data():
    return [
        ("I love Mumbai".split(), ["PRON", "VERB", "LOC"]),
        ("She lives in Pune".split(), ["PRON", "VERB", "PREP", "LOC"]),
        ("He works at Google".split(), ["PRON", "VERB", "PREP", "ORG"])
    ]

def build_vocab(data):
    word_to_ix = {}
    tag_to_ix = {}

    for sentence, tags in data:
        for word in sentence:
            if word not in word_to_ix:
                word_to_ix[word] = len(word_to_ix)

        for tag in tags:
            if tag not in tag_to_ix:
                tag_to_ix[tag] = len(tag_to_ix)

    ix_to_tag = {v: k for k, v in tag_to_ix.items()}

    return word_to_ix, tag_to_ix, ix_to_tag

def prepare_sequence(seq, to_ix):
    return torch.tensor([to_ix[w] for w in seq], dtype=torch.long)

class LSTMTagger(nn.Module):
    def __init__(self, vocab_size, tagset_size, embedding_dim=8, hidden_dim=16):
        super(LSTMTagger, self).__init__()

        self.embedding = nn.Embedding(vocab_size, embedding_dim)
        self.lstm = nn.LSTM(embedding_dim, hidden_dim)

        self.fc = nn.Linear(hidden_dim, tagset_size)

    def forward(self, sentence):
        embeds = self.embedding(sentence)
        lstm_out, _ = self.lstm(embeds.view(len(sentence), 1, -1))
        outputs = self.fc(lstm_out.view(len(sentence), -1))
        tag_scores = torch.log_softmax(outputs, dim=1)
        return tag_scores

def train_model(model, data, word_to_ix, tag_to_ix, epochs=100, lr=0.1):
    loss_function = nn.NLLLoss()
    optimizer = optim.SGD(model.parameters(), lr=lr)

    for epoch in range(epochs):
        total_loss = 0

        for sentence, tags in data:
            model.zero_grad()

            inputs = prepare_sequence(sentence, word_to_ix)
            targets = prepare_sequence(tags, tag_to_ix)

            outputs = model(inputs)
            loss = loss_function(outputs, targets)

            loss.backward()
            optimizer.step()

            total_loss += loss.item()

        if (epoch + 1) % 20 == 0:
            print(f"Epoch {epoch+1}, Loss: {total_loss:.4f}")

def predict(model, sentence, word_to_ix, ix_to_tag):
    with torch.no_grad():
        inputs = prepare_sequence(sentence.split(), word_to_ix)
        outputs = model(inputs)

        predicted_indices = torch.argmax(outputs, dim=1).tolist()
        predicted_tags = [ix_to_tag[ix] for ix in predicted_indices]

        return predicted_tags

def main():
    # Load data
    data = load_data()

    # Build vocab
    word_to_ix, tag_to_ix, ix_to_tag = build_vocab(data)

    # Initialize model
    model = LSTMTagger(
        vocab_size=len(word_to_ix),
        tagset_size=len(tag_to_ix)
    )

    # Train
    train_model(model, data, word_to_ix, tag_to_ix)

    # Test
    test_sentence = "She works at Mumbai"
    predicted_tags = predict(model, test_sentence, word_to_ix, ix_to_tag)

    print("\nTest Sentence:", test_sentence.split())
    print("Predicted Tags:", predicted_tags)

if __name__ == "__main__":
    main()

Epoch 20, Loss: 4.1791
Epoch 40, Loss: 3.4264
Epoch 60, Loss: 2.3889
Epoch 80, Loss: 1.3761
Epoch 100, Loss: 0.7556

Test Sentence: ['She', 'works', 'at', 'Mumbai']
Predicted Tags: ['PRON', 'VERB', 'PREP', 'LOC']
